# Notebook 08 — Research Synthesis & Factor Extraction

This notebook converts the grounded RAG evidence from Notebook 07 into structured quantitative-research insights.

**Input**
- `data/vector_store/rag_last_response.json`
- `data/vector_store/quant_research_metadata.json`

**Output**
- Research synthesis
- Extracted quantitative factors
- Evidence / citation mapping
- `data/outputs/research_synthesis.json`

**Principle:** every extracted insight must remain traceable to the retrieved PDF/chunk evidence.


In [4]:
# 01 — Imports and paths

import os
import json
import re
from pathlib import Path
from dotenv import load_dotenv, find_dotenv
from google import genai

# Project root = parent of notebooks/
PROJECT_ROOT = Path.cwd().parent

RAG_RESPONSE_PATH = (
    PROJECT_ROOT / "data" / "vector_store" / "rag_last_response.json"
)

METADATA_PATH = (
    PROJECT_ROOT / "data" / "vector_store" / "quant_research_metadata.json"
)

OUTPUT_DIR = PROJECT_ROOT / "data" / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("RAG response:", RAG_RESPONSE_PATH)
print("RAG response exists:", RAG_RESPONSE_PATH.exists())
print("Metadata exists:", METADATA_PATH.exists())

Project root: c:\Users\MD RAJA\Quant_Research_Intelligence_System
RAG response: c:\Users\MD RAJA\Quant_Research_Intelligence_System\data\vector_store\rag_last_response.json
RAG response exists: True
Metadata exists: True


In [5]:
# 02 — Load Gemini configuration

env_path = find_dotenv()

if not env_path:
    raise FileNotFoundError(".env file not found.")

load_dotenv(env_path, override=True)

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")
LLM_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.5-flash-lite")

if not GEMINI_API_KEY:
    raise ValueError("GEMINI_API_KEY is missing from .env")

client = genai.Client(api_key=GEMINI_API_KEY)

print("Gemini client: READY")
print("Model:", LLM_MODEL)


Gemini client: READY
Model: gemini-3.5-flash-lite


In [6]:
# 03 — Load Notebook 07 response and chunk metadata

with open(RAG_RESPONSE_PATH, "r", encoding="utf-8") as f:
    rag_result = json.load(f)

with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print("RAG response keys:", list(rag_result.keys()))
print("Metadata records:", len(metadata) if isinstance(metadata, list) else "dictionary")


RAG response keys: ['query', 'status', 'answer', 'citations', 'traceability', 'guardrail', 'llm_model']
Metadata records: 1141


In [7]:
# 04 — Build chunk lookup

# Support both list-style and dictionary-style metadata files.
if isinstance(metadata, list):
    chunk_records = metadata
elif isinstance(metadata, dict):
    if "records" in metadata:
        chunk_records = metadata["records"]
    elif "metadata" in metadata:
        chunk_records = metadata["metadata"]
    else:
        chunk_records = list(metadata.values())
else:
    chunk_records = []

chunk_lookup = {}

for record in chunk_records:
    if isinstance(record, dict):
        chunk_id = record.get("chunk_id")
        if chunk_id:
            chunk_lookup[chunk_id] = record

print("Chunk lookup:", len(chunk_lookup))


Chunk lookup: 1141


In [14]:
# 05 — Recover the evidence used by Notebook 07

traceability = rag_result.get("traceability", [])
citations = rag_result.get("citations", [])

print("Citations:", citations)
print("Traceability records:", len(traceability))

evidence_records = []

for item in traceability:
    if not isinstance(item, dict):
        continue

    chunk_id = item.get("chunk_id")

    record = chunk_lookup.get(chunk_id, {}).copy()

    # Notebook 07 stores the citation as "citation"
    record.update({
        "citation_id": item.get("citation"),
        "chunk_id": chunk_id,
        "source_file": item.get("source_file"),
        "pages": item.get("pages"),
        "section_path": item.get("section_path"),
        "element_ids": item.get("element_ids", []),
        "figure_ids": item.get("figure_ids", []),
        "table_ids": item.get("table_ids", []),
        "formula_ids": item.get("formula_ids", []),
        "image_paths": item.get("image_paths", []),
        "evidence": item.get("evidence", []),
    })

    evidence_records.append(record)

print("\nEvidence recovered:", len(evidence_records))

for record in evidence_records:
    print(
        f"{record['citation_id']} → "
        f"{record['source_file']} → "
        f"pages {record['pages']} → "
        f"{record['chunk_id']}"
    )

Citations: ['S5', 'S6']
Traceability records: 2

Evidence recovered: 2
S5 → ssrn-5231379.pdf → pages [4] → chunk_d9600aaaec5dcae2
S6 → ssrn-5231379.pdf → pages [5] → chunk_f1c8dbdb6d16fb1b


In [15]:
# 06 — Prepare evidence for synthesis

def get_record_text(record):
    return (
        record.get("text")
        or record.get("retrieval_text")
        or record.get("content")
        or ""
    )

evidence_blocks = []

for record in evidence_records:
    citation = record.get("citation_id") or record.get("source_id") or "S?"
    source = record.get("source_file", "unknown")
    pages = record.get("pages", [])
    section = record.get("section_path", [])
    text = get_record_text(record)

    evidence_blocks.append(
        f"[{citation}]\n"
        f"Source: {source}\n"
        f"Pages: {pages}\n"
        f"Section: {section}\n"
        f"Chunk: {record.get('chunk_id')}\n"
        f"Evidence:\n{text}"
    )

evidence_text = "\n\n".join(evidence_blocks)

print("Evidence characters:", len(evidence_text))
print(evidence_text[:3000])


Evidence characters: 3664
[S5]
Source: ssrn-5231379.pdf
Pages: [4]
Section: ['1. Introduction']
Chunk: chunk_d9600aaaec5dcae2
Evidence:
These mechanisms have non-monotonic effects on overall trading aggressiveness of traders. When q is low, the strategic myopia effect dominates at all levels of γ, leading to a decrease in overall aggressiveness. When q is high, the positive effects dominate and trading aggressiveness increases with γ. At intermediate values of q, the relationship is Ushaped: overall aggressiveness first decreases, and then increases. This result shows that the impact of growing quant participation in markets hinges on both their informational advantage and their ability to account for strategic feedback. Since prices are determined by order flow in our model, changes in trading behavior naturally affect price efficiency. Following Kyle (1985) and related work (e.g., Subrahmanyam 1991; Holden and Subrahmanyam 1992), we define price efficiency as the inverse of the condi

In [16]:
# 07 — Structured research synthesis prompt

SYSTEM_PROMPT = """
You are a quantitative-finance research synthesis assistant.

Use ONLY the supplied evidence.
Do not introduce facts that are not supported by the evidence.

For every factor or research claim:
- preserve the terminology used by the papers
- explain the economic/research idea briefly
- include a formula only when the evidence provides or supports it
- attach the exact evidence citation such as [S5]
- distinguish directly stated facts from cautious synthesis across sources

Return valid JSON only.
"""

USER_PROMPT = f"""
Synthesize the following retrieved quantitative-finance research evidence.

EVIDENCE:
{evidence_text}

Return this JSON structure:

{{
  "research_theme": "...",
  "key_findings": [
    {{
      "finding": "...",
      "evidence": ["S1", "S2"]
    }}
  ],
  "factors": [
    {{
      "factor_name": "...",
      "definition": "...",
      "economic_intuition": "...",
      "formula": "... or null",
      "evidence": ["S1"]
    }}
  ],
  "relationships": [
    {{
      "relationship": "...",
      "evidence": ["S1", "S2"]
    }}
  ],
  "limitations": [
    {{
      "limitation": "...",
      "evidence": ["S1"]
    }}
  ]
}}
"""

print("Synthesis prompt prepared.")


Synthesis prompt prepared.


In [17]:
# 08 — Generate structured research synthesis

response = client.models.generate_content(
    model=LLM_MODEL,
    contents=USER_PROMPT,
    config=genai.types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        temperature=0.1,
        max_output_tokens=1800,
    ),
)

raw_synthesis = (response.text or "").strip()

# Remove accidental markdown fences if the model returns them.
clean_synthesis = re.sub(
    r"^```(?:json)?\s*|\s*```$",
    "",
    raw_synthesis,
    flags=re.IGNORECASE | re.DOTALL,
).strip()

synthesis = json.loads(clean_synthesis)

print(json.dumps(synthesis, indent=2, ensure_ascii=False))


{
  "research_theme": "The impact of quantitative investing and strategic feedback on trading aggressiveness and price efficiency",
  "key_findings": [
    {
      "finding": "The impact of growing quant participation on overall trading aggressiveness and price efficiency depends non-monotonically on signal quality (q) and the quantitative investing level (γ). When q is low, aggressiveness and price efficiency decrease with γ due to the dominance of the strategic myopia effect. When q is high, trading aggressiveness and price efficiency increase with γ. At intermediate values of q, the relationship is U-shaped.",
      "evidence": [
        "S5"
      ]
    },
    {
      "finding": "Introducing a competition effect—where both firms have quant and discretionary departments—shows that as firm B's quantitative investing level increases, it competes more aggressively on additional signals, causing firm A's quant department to reduce its aggressiveness on that signal. The impact of one fir

In [13]:
print("RAG keys:")
print(rag_result.keys())

print("\nFull RAG result structure:")
print(json.dumps(rag_result, indent=2)[:8000])

RAG keys:
dict_keys(['query', 'status', 'answer', 'citations', 'traceability', 'guardrail', 'llm_model'])

Full RAG result structure:
{
  "query": "What does the retrieved research say about quantitative trading and market efficiency?",
  "status": "success",
  "answer": "Based on the provided research context, quantitative trading affects market efficiency depending on the quality of the quants' information and their level of participation:\n\n* **Information Quality:** When quants' information quality is low, an increase in quantitative trading participation deteriorates price efficiency [S1, S5]. When their information quality is high, it improves price efficiency [S1, S5]. At moderate information quality levels, the relationship is U-shaped: price efficiency initially decreases, and then improves once the quantitative investing level exceeds a threshold [S5]. \n* **Underlying Mechanisms:** Quants are characterized by two opposing forces\u2014superior information processing capacity

In [18]:
# 09 — Validate citation traceability

valid_citations = {
    str(record.get("citation_id"))
    for record in evidence_records
    if record.get("citation_id") is not None
}

def extract_citations(obj):
    found = set()

    if isinstance(obj, dict):
        for value in obj.values():
            found.update(extract_citations(value))

    elif isinstance(obj, list):
        for value in obj:
            found.update(extract_citations(value))

    elif isinstance(obj, str):
        found.update(re.findall(r"\bS\d+\b", obj))

    return found

used_citations = extract_citations(synthesis)
unresolved = sorted(used_citations - valid_citations)

print("Valid evidence citations:", sorted(valid_citations))
print("Citations used by synthesis:", sorted(used_citations))
print("Unresolved citations:", unresolved)

assert not unresolved, f"Unresolved citations found: {unresolved}"

print("\nCitation traceability QA: PASS")


Valid evidence citations: ['S5', 'S6']
Citations used by synthesis: ['S5', 'S6']
Unresolved citations: []

Citation traceability QA: PASS


In [19]:
# 10 — Display extracted factors

factors = synthesis.get("factors", [])

print("Extracted factors:", len(factors))

for i, factor in enumerate(factors, start=1):
    print(f"\nFactor {i}: {factor.get('factor_name')}")
    print("Definition:", factor.get("definition"))
    print("Economic intuition:", factor.get("economic_intuition"))
    print("Formula:", factor.get("formula"))
    print("Evidence:", factor.get("evidence"))


Extracted factors: 3

Factor 1: Quantitative investing level (γ)
Definition: The level of quantitative investing within a firm or market participant.
Economic intuition: Captures the extent to which a firm relies on quantitative methods versus discretionary approaches, affecting information processing and strategic behavior.
Formula: None
Evidence: ['S5', 'S6']

Factor 2: Signal quality (q)
Definition: The quality of the information signal processed by market participants.
Economic intuition: Determines the informational advantage held by quantitative investors, influencing whether processing or myopia effects dominate trading aggressiveness and price efficiency.
Formula: None
Evidence: ['S5', 'S6']

Factor 3: Degree of strategic myopia (h)
Definition: A parameter representing the degree of strategic myopia, where h = 1 corresponds to fully myopic and h = 0 corresponds to a fully strategic quant.
Economic intuition: Governs how myopic quants are regarding strategic feedback; lower valu

In [20]:
# 11 — Save structured research synthesis

output = {
    "source_rag_response": str(RAG_RESPONSE_PATH),
    "model": LLM_MODEL,
    "citations_used": sorted(used_citations),
    "evidence_records": [
        {
            "citation_id": r.get("citation_id"),
            "chunk_id": r.get("chunk_id"),
            "source_file": r.get("source_file"),
            "pages": r.get("pages"),
            "section_path": r.get("section_path"),
            "element_ids": r.get("element_ids"),
            "figure_ids": r.get("figure_ids"),
            "table_ids": r.get("table_ids"),
            "formula_ids": r.get("formula_ids"),
        }
        for r in evidence_records
    ],
    "synthesis": synthesis,
}

OUTPUT_PATH = OUTPUT_DIR / "research_synthesis.json"

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(output, f, indent=2, ensure_ascii=False)

print("Saved:", OUTPUT_PATH)


Saved: c:\Users\MD RAJA\Quant_Research_Intelligence_System\data\outputs\research_synthesis.json


## Notebook 08 → Notebook 09 Evaluation

Notebook 08 produces:

```text
Research Theme
    ↓
Key Findings
    ↓
Quantitative Factors
    ├── Definition
    ├── Economic Intuition
    ├── Formula
    └── Evidence [Sx]
            ↓
      PDF / Page / Chunk
```

**notebook  09 will be evaluation.

